### ЗАДАЧА: Учёт посещаемости мероприятий

Есть список регистраций участников на внутренние мероприятия.
Нужно обработать его через классы и собрать отчёт по событиям и участникам.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Класс `AttendanceRecord` с полями `person`, `event_name`, `day`, `status`.

2. В `AttendanceRecord` реализовать:
   - метод `to_dict()`
   - `@classmethod from_tuple(cls, row)`.

3. В `AttendanceRecord.from_tuple(cls, row)`:
   - если формат строки неверный, вернуть `(None, "bad format")`
   - если статус не входит в допустимые, вернуть `(None, "bad status")`
   - если запись корректна, вернуть `(record, "")`.

4. Класс `AttendanceRegistry`:
   - хранит записи в списке `self.records`
   - хранит невалидные строки в списке `self.invalid_rows`
   - хранит дубли в списке `self.duplicates`.

5. В `AttendanceRegistry` реализовать методы:
   - `add_record(record)`
   - `load_rows(rows)`
   - `event_stats()`
   - `person_history(person)`
   - `active_events()`
   - `build_report()`.

6. При добавлении записи:
   - считать дублем совпадение `person`, `event_name`, `day`
   - дубль не добавлять
   - информацию о дубле сохранять в `self.duplicates`.

7. После загрузки данных вывести:
   - количество корректных записей
   - количество ошибок
   - количество дублей
   - статистику по мероприятиям
   - множество активных мероприятий.


In [ ]:
rows = [
    ("Алиса", "Python Meetup", "Mon", "registered"),
    ("Алиса", "Python Meetup", "Mon", "attended"),
    ("Боб", "Data Talk", "Tue", "registered"),
    ("Кира", "Data Talk", "Tue", "attended"),
    ("Данил", "Design Review", "Wed", "cancelled"),
    ("Ева", "Python Meetup", "Thu", "waiting"),
    ("Боб", "Data Talk", "Tue", "registered")
]


class AttendanceRecord:
    # TODO:
    # реализуй __init__(person, event_name, day, status)

    def to_dict(self):
        # TODO:
        # вернуть словарь с полями person, event_name, day, status
        pass

    @classmethod
    def from_tuple(cls, row):
        # TODO:
        # 1) проверить длину row
        # 2) распаковать значения
        # 3) проверить статус
        # 4) вернуть (record, "") или (None, reason)
        pass


class AttendanceRegistry:
    def __init__(self):
        # TODO:
        # self.records = []
        # self.invalid_rows = []
        # self.duplicates = []
        pass

    def add_record(self, record):
        # TODO:
        # проверить, есть ли уже запись с теми же person, event_name, day
        # если да:
        #    добавить (record.person, record.event_name, record.day) в self.duplicates
        #    не добавлять запись
        # иначе добавить record в self.records
        pass

    def load_rows(self, rows):
        # TODO:
        # пройти по rows
        # вызвать AttendanceRecord.from_tuple(row)
        # если запись невалидна -> self.invalid_rows.append((row, reason))
        # иначе -> self.add_record(record)
        pass

    def event_stats(self):
        # TODO:
        # вернуть словарь статистики по мероприятиям и статусам
        pass

    def person_history(self, person):
        # TODO:
        # вернуть список записей выбранного участника
        pass

    def active_events(self):
        # TODO:
        # вернуть множество событий, где есть хотя бы один status == 'attended'
        pass

    def build_report(self):
        # TODO:
        # вернуть словарь с ключами:
        # total_valid_records, total_invalid_rows,
        # total_duplicates, event_stats, active_events
        pass


registry = AttendanceRegistry()
registry.load_rows(rows)
report = registry.build_report()

print("Корректных записей:", report["total_valid_records"])
print("Ошибок:", report["total_invalid_rows"])
print("Дублей:", report["total_duplicates"])
print("Статистика мероприятий:", report["event_stats"])
print("Активные мероприятия:", report["active_events"])


In [ ]:
# Решение


rows = [
    ("Алиса", "Python Meetup", "Mon", "registered"),
    ("Алиса", "Python Meetup", "Mon", "attended"),
    ("Боб", "Data Talk", "Tue", "registered"),
    ("Кира", "Data Talk", "Tue", "attended"),
    ("Данил", "Design Review", "Wed", "cancelled"),
    ("Ева", "Python Meetup", "Thu", "waiting"),
    ("Боб", "Data Talk", "Tue", "registered")
]


class AttendanceRecord:
    def __init__(self, person, event_name, day, status):
        self.person = person
        self.event_name = event_name
        self.day = day
        self.status = status

    def to_dict(self):
        return {
            "person": self.person,
            "event_name": self.event_name,
            "day": self.day,
            "status": self.status,
        }

    @classmethod
    def from_tuple(cls, row):
        if len(row) != 4:
            return None, "bad format"

        person, event_name, day, status = row
        allowed_statuses = {"registered", "attended", "cancelled"}

        if status not in allowed_statuses:
            return None, "bad status"

        return cls(person, event_name, day, status), ""


class AttendanceRegistry:
    def __init__(self):
        self.records = []
        self.invalid_rows = []
        self.duplicates = []

    def add_record(self, record):
        for existing in self.records:
            if (
                existing.person == record.person
                and existing.event_name == record.event_name
                and existing.day == record.day
            ):
                self.duplicates.append((record.person, record.event_name, record.day))
                return

        self.records.append(record)

    def load_rows(self, rows):
        for row in rows:
            record, reason = AttendanceRecord.from_tuple(row)
            if record is None:
                self.invalid_rows.append((row, reason))
            else:
                self.add_record(record)

    def event_stats(self):
        stats = {}
        for record in self.records:
            if record.event_name not in stats:
                stats[record.event_name] = {
                    "registered": 0,
                    "attended": 0,
                    "cancelled": 0,
                }
            stats[record.event_name][record.status] += 1
        return stats

    def person_history(self, person):
        return [record for record in self.records if record.person == person]

    def active_events(self):
        result = set()
        for record in self.records:
            if record.status == "attended":
                result.add(record.event_name)
        return result

    def build_report(self):
        return {
            "total_valid_records": len(self.records),
            "total_invalid_rows": len(self.invalid_rows),
            "total_duplicates": len(self.duplicates),
            "event_stats": self.event_stats(),
            "active_events": self.active_events(),
        }


registry = AttendanceRegistry()
registry.load_rows(rows)
report = registry.build_report()

print("Корректных записей:", report["total_valid_records"])
print("Ошибок:", report["total_invalid_rows"])
print("Дублей:", report["total_duplicates"])
print("Статистика мероприятий:", report["event_stats"])
print("Активные мероприятия:", report["active_events"])
